# Notebook 01 · Data Quality & Preparation

## Objetivo

Evaluar la calidad de los datos almacenados en la base de datos SQLite para garantizar que el análisis posterior se realiza sobre información consistente y fiable.

## Problema que resuelve

Antes de realizar cualquier análisis estadístico o generar nuevas variables, es necesario verificar que la estructura de los datos es consistente y que no existen incidencias que puedan afectar a la calidad del análisis.

## Alcance

Este notebook incluye:

- Conexión a la base de datos SQLite.
- Carga de las tablas del modelo relacional.
- Inspección inicial de los datos.
- Validación de la calidad de los datos.
- Comprobación de la consistencia del modelo relacional.
- Estadísticos descriptivos básicos.

Este notebook no incluye:

- Análisis de negocio.
- KPIs.
- Rankings.
- Visualizaciones estadísticas.
- Feature Engineering.


# 2. Importación de librerías

En esta sección se importan las librerías necesarias para establecer la conexión con la base de datos SQLite y realizar el análisis de calidad de los datos.

Durante esta fase se utilizarán únicamente las librerías estrictamente necesarias, incorporando nuevas dependencias únicamente cuando aporten un valor al análisis.

In [1]:
# Librerías estándar
import sqlite3
from pathlib import Path

# Librerías de terceros
import pandas as pd

In [2]:
# ==========================================================
# Configuración
# ==========================================================

# Ruta raíz del proyecto
PROJECT_ROOT = Path.cwd().parent

# Ruta de la base de datos SQLite
DATABASE_PATH = PROJECT_ROOT / "data" / "raw" / "database" / "bank_sqlite.db"

# Configuración de pandas
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", "{:.2f}".format)


In [3]:
print(PROJECT_ROOT)
print(DATABASE_PATH)
print(DATABASE_PATH.exists())

c:\Users\Lenovo\OneDrive\Documentos\Data_Projects\retail-banking-customer-intelligence
c:\Users\Lenovo\OneDrive\Documentos\Data_Projects\retail-banking-customer-intelligence\data\raw\database\bank_sqlite.db
True


# 3. Conexión a la base de datos SQLite

En esta sección se establece la conexión con la base de datos SQLite que contiene toda la información utilizada en el proyecto.

Una conexión correcta garantiza que las tablas puedan cargarse posteriormente para realizar la validación de la calidad de los datos.

In [4]:
# ==========================================================
# Conexión a la base de datos
# ==========================================================

try:
    connection = sqlite3.connect(DATABASE_PATH)
    print("Conexión a la base de datos establecida.")

except sqlite3.Error as error:
    print(f"Error al conectar a la base de datos: {error}")
        

Conexión a la base de datos establecida.


# 4. Carga de las tablas

En esta sección se cargan todas las tablas de la base de datos SQLite en DataFrames de pandas.

El objetivo es disponer de una copia de trabajo de cada tabla para realizar las comprobaciones de calidad y consistencia durante el resto del notebook.

In [5]:
# Obtener el nombre de todas las tablas de la base de datos

query = """
SELECT name
FROM sqlite_master
WHERE type='table'
ORDER BY name;
"""

tables = pd.read_sql(query, connection)

tables

,name
0,accounts
1,branches
2,cards
3,customers
4,loans
5,merchants
6,transactions


In [9]:
accounts_df = pd.read_sql("SELECT * FROM accounts", connection)
branches_df = pd.read_sql("SELECT * FROM branches", connection)
cards_df = pd.read_sql("SELECT * FROM cards", connection)
customers_df = pd.read_sql("SELECT * FROM customers", connection)
loans_df = pd.read_sql("SELECT * FROM loans", connection)
merchants_df = pd.read_sql("SELECT * FROM merchants", connection)
transactions_df = pd.read_sql("SELECT * FROM transactions", connection)

print("Tablas cargadas correctamente:\n")

print(f"• Accounts: {accounts_df.shape}")
print(f"• Branches: {branches_df.shape}")
print(f"• Cards: {cards_df.shape}")
print(f"• Customers: {customers_df.shape}")
print(f"• Loans: {loans_df.shape}")
print(f"• Merchants: {merchants_df.shape}")
print(f"• Transactions: {transactions_df.shape}")


Tablas cargadas correctamente:

• Accounts: (75000, 5)
• Branches: (500, 5)
• Cards: (100000, 4)
• Customers: (50000, 7)
• Loans: (30000, 5)
• Merchants: (5000, 3)
• Transactions: (1000000, 5)


### Conclusión

La carga de datos se ha realizado correctamente. Todas las tablas del modelo relacional están disponibles para el análisis y presentan un volumen de información suficiente para desarrollar el proyecto.

En total, la base de datos contiene siete tablas relacionadas que representan clientes, cuentas, tarjetas, préstamos, sucursales, comercios y transacciones.

# 5. Inspección inicial de las tablas

Antes de evaluar la calidad de los datos, es necesario comprender la estructura de cada tabla.

En esta sección se revisarán:

- Las primeras filas de cada tabla.
- La estructura de columnas.
- Los tipos de datos.
- El número de registros y columnas.

Esta inspección permite detectar posibles inconsistencias antes de realizar comprobaciones más específicas.

In [20]:
# Diccionario con todas las tablas del proyecto
tables = {
    "Accounts": accounts_df,
    "Branches": branches_df,
    "Cards": cards_df,
    "Customers": customers_df,
    "Loans": loans_df,
    "Merchants": merchants_df,
    "Transactions": transactions_df,
}

In [21]:
for table_name, dataframe in tables.items():

    print("=" * 80)
    print(f"TABLA: {table_name}")
    print("=" * 80)

    display(dataframe.head())

    print("\nInformación de la tabla:\n")
    dataframe.info()

    print("\n")

TABLA: Accounts


,account_id,customer_id,account_type,balance_usd,open_date
0,ACC000NV80W3W25,CUSEU25WUTM7HK9,Checking,27307.96,2022-03-17 08:31:20
1,ACC002FHD9SACSA,CUSLTOG1EPNYETU,Checking,157104.62,2021-10-28 12:55:25
2,ACC002XR7B6XDZS,CUSZ6YW1EL8RT7P,Business,50462.17,2021-07-14 08:43:28
3,ACC003O7MY9B1ZN,CUSRPHL7ZTPEAXW,Checking,35793.36,2025-01-19 12:33:15
4,ACC003R49JACSPQ,CUSHX3KW6F31CBE,Checking,127048.71,2019-03-16 11:05:54



Información de la tabla:

<class 'pandas.DataFrame'>
RangeIndex: 75000 entries, 0 to 74999
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   account_id    75000 non-null  str    
 1   customer_id   75000 non-null  str    
 2   account_type  75000 non-null  str    
 3   balance_usd   75000 non-null  float64
 4   open_date     75000 non-null  str    
dtypes: float64(1), str(4)
memory usage: 2.9 MB


TABLA: Branches


,branch_id,branch_name,manager_name,city,country
0,BRN01ZNN78I3V3D,East Nicholas Branch,Diane Larson,None,None
1,BRN0307FBDU2Q4K,North Dustin Branch,Mary Hernandez,None,None
2,BRN045ZQQK60738,South Reginald Branch,Susan Spencer,None,None
3,BRN0585SDS271EC,North Amber Branch,Bruce Chapman,None,None
4,BRN06FSFLBAAHKI,Lake Kimberly Branch,Melissa Fleming,None,None



Información de la tabla:

<class 'pandas.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 5 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   branch_id     500 non-null    str   
 1   branch_name   500 non-null    str   
 2   manager_name  500 non-null    str   
 3   city          0 non-null      object
 4   country       0 non-null      object
dtypes: object(2), str(3)
memory usage: 19.7+ KB


TABLA: Cards


...


Información de la tabla:

<class 'pandas.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   loan_id        30000 non-null  str    
 1   customer_id    30000 non-null  str    
 2   loan_amount    30000 non-null  float64
 3   interest_rate  30000 non-null  float64
 4   start_date     30000 non-null  str    
dtypes: float64(2), str(3)
memory usage: 1.1 MB


TABLA: Merchants


,merchant_id,merchant_name,city
0,MER008Y63JK0BUX,Gregory-Meyer,Port Cindy
1,MER00BRU6D2NHTI,"Randall, Gibbs and Dennis",Davidmouth
2,MER00C6HH0ULQQD,Clayton Inc,Port Kevinbury
3,MER00CEA2H7VS72,"Baker, Michael and Burns",West Tyler
4,MER00GJVJ7OGPL9,Owens Group,South Markport



Información de la tabla:

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 3 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   merchant_id    5000 non-null   str  
 1   merchant_name  5000 non-null   str  
 2   city           5000 non-null   str  
dtypes: str(3)
memory usage: 117.3 KB


TABLA: Transactions


,transaction_id,account_id,merchant_id,amount_usd,transaction_date
0,TXN00005HFWN2XNY3,ACCP1YZO9D6NWHS,MER1ECHZUV9E1WF,9239.92,2022-01-14 04:58:46
1,TXN00009897MR3AF0,ACC39NBWQ11B0YW,MERBTO90OBMKTWM,8261.44,2021-02-20 00:12:15
2,TXN0002DXC6LENBZN,ACCVEP6O0JTRPHJ,MERI68JP0JLHMNF,127.38,2024-06-02 11:20:37
3,TXN0003B7W0PHYMBL,ACC9DAR8N097DNX,MERN7Z2NR7VK3E6,501.59,2020-03-08 22:36:55
4,TXN0003LEFVFA0V95,ACCNULMR5ZNMBK9,MER5284SUXHD16Q,1535.35,2024-09-01 06:49:37



Información de la tabla:

<class 'pandas.DataFrame'>
RangeIndex: 1000000 entries, 0 to 999999
Data columns (total 5 columns):
 #   Column            Non-Null Count    Dtype  
---  ------            --------------    -----  
 0   transaction_id    1000000 non-null  str    
 1   account_id        1000000 non-null  str    
 2   merchant_id       1000000 non-null  str    
 3   amount_usd        1000000 non-null  float64
 4   transaction_date  1000000 non-null  str    
dtypes: float64(1), str(4)
memory usage: 38.1 MB




### Conclusión

La inspección inicial permite verificar que todas las tablas han sido cargadas correctamente y que su estructura es coherente con el modelo de datos esperado.

La revisión de las primeras filas, las dimensiones y los tipos de datos proporciona una visión general del contenido de cada tabla y constituye el punto de partida para las comprobaciones de calidad que se realizarán en las siguientes secciones.

# 6. Validación de valores nulos

Los valores nulos pueden afectar tanto a la calidad del análisis como a la construcción del modelo analítico.

En esta sección se evaluará la existencia de valores nulos en todas las tablas de la base de datos para identificar posibles incidencias que requieran tratamiento en fases posteriores del proyecto.

El objetivo no es corregir los valores nulos, sino detectar su presencia y cuantificar su impacto.

In [18]:
# ==========================================================
# Validación de valores nulos
# ==========================================================

for table_name, dataframe in tables.items():

    print("=" * 80)
    print(f"TABLA: {table_name}")
    print("=" * 80)

    missing = dataframe.isnull().sum()

    missing = missing[missing > 0]

    if missing.empty:
        print("✅ No se han encontrado valores nulos.\n")

    else:

        missing_df = pd.DataFrame({
            "Valores nulos": missing,
            "Porcentaje (%)": (missing / len(dataframe) * 100).round(2)
        })

        display(missing_df)

    print()

TABLA: Accounts
✅ No se han encontrado valores nulos.


TABLA: Branches


,Valores nulos,Porcentaje (%)
city,500,100.00
country,500,100.00



TABLA: Cards
✅ No se han encontrado valores nulos.


TABLA: Customers
✅ No se han encontrado valores nulos.


TABLA: Loans
✅ No se han encontrado valores nulos.


TABLA: Merchants
✅ No se han encontrado valores nulos.


TABLA: Transactions
✅ No se han encontrado valores nulos.




### Conclusión

La revisión de valores nulos muestra una calidad de datos elevada en la mayor parte del modelo relacional.

Únicamente la tabla **Branches** presenta valores nulos, concretamente en las columnas **city** y **country**, donde el 100 % de los registros carecen de información.

En las siguientes fases se evaluará si estas columnas son relevantes para el análisis o si pueden excluirse sin impacto sobre los objetivos del proyecto.

# 7. Validación de registros duplicados

Los registros duplicados pueden introducir sesgos en los análisis y afectar a la calidad de los indicadores obtenidos.

En esta sección se comprobará la existencia de filas duplicadas en cada una de las tablas del modelo de datos para identificar posibles problemas de integridad.

In [22]:
# ==========================================================
# Validación de registros duplicados
# ==========================================================

for table_name, dataframe in tables.items():

    duplicated = dataframe.duplicated().sum()

    print("=" * 80)
    print(f"📋 TABLA: {table_name}")
    print("=" * 80)

    print(f"Registros duplicados: {duplicated:,}\n")

📋 TABLA: Accounts
Registros duplicados: 0

📋 TABLA: Branches
Registros duplicados: 0

📋 TABLA: Cards
Registros duplicados: 0

📋 TABLA: Customers
Registros duplicados: 0

📋 TABLA: Loans
Registros duplicados: 0

📋 TABLA: Merchants
Registros duplicados: 0

📋 TABLA: Transactions
Registros duplicados: 0



### Conclusión

No se han detectado registros duplicados en ninguna de las tablas que componen el modelo relacional.

Este resultado indica que el conjunto de datos mantiene la unicidad esperada de sus registros y no presenta incidencias de calidad relacionadas con duplicados a nivel de fila.

# 8. Validación de la integridad referencial

La integridad referencial garantiza que las relaciones entre las distintas tablas del modelo de datos sean consistentes.

En esta sección se comprobará que todas las claves foráneas hacen referencia a registros existentes en sus correspondientes tablas maestras.

Una correcta integridad referencial asegura la consistencia del modelo relacional y evita errores durante el análisis posterior.

In [23]:
# ==========================================================
# Validación de la integridad referencial
# ==========================================================

relations = [

    ("Accounts", accounts_df, "customer_id", customers_df, "customer_id"),

    ("Cards", cards_df, "account_id", accounts_df, "account_id"),

    ("Loans", loans_df, "customer_id", customers_df, "customer_id"),

    ("Transactions", transactions_df, "account_id", accounts_df, "account_id"),

]

for child_name, child_df, child_key, parent_df, parent_key in relations:

    invalid_records = (~child_df[child_key].isin(parent_df[parent_key])).sum()

    print("=" * 80)
    print(f"📋 {child_name}")
    print("=" * 80)

    print(f"Registros sin correspondencia: {invalid_records:,}\n")

📋 Accounts
Registros sin correspondencia: 0

📋 Cards
Registros sin correspondencia: 0

📋 Loans
Registros sin correspondencia: 0

📋 Transactions
Registros sin correspondencia: 0



### Conclusión

La validación de la integridad referencial confirma que todas las relaciones entre las tablas del modelo de datos son consistentes.

No se han detectado registros huérfanos en ninguna de las claves foráneas analizadas, lo que garantiza que las relaciones entre clientes, cuentas, tarjetas, préstamos y transacciones mantienen la integridad esperada.

Este resultado proporciona una base sólida para continuar con el análisis estadístico y la construcción del modelo analítico.

# 9. Estadísticos descriptivos de las variables numéricas

Una vez validada la calidad del modelo relacional, se realiza una revisión descriptiva de las principales variables numéricas.

El objetivo de esta sección no es obtener conclusiones de negocio, sino verificar que los rangos de valores, las medidas de tendencia central y la dispersión son coherentes con la naturaleza de los datos.

Esta revisión también permite identificar posibles valores anómalos que puedan requerir un análisis más detallado en las siguientes fases del proyecto.

In [25]:
# ==========================================================
# Estadísticos descriptivos
# ==========================================================

for table_name, dataframe in tables.items():

    # Obtener únicamente las columnas numéricas
    numeric_df = dataframe.select_dtypes(include=["number"])

    # Si la tabla no contiene variables numéricas, pasar a la siguiente
    if numeric_df.empty:
        continue

    print("=" * 80)
    print(f"📊 TABLA: {table_name}")
    print("=" * 80)

    display(numeric_df.describe().T.round(2))

    print()

📊 TABLA: Accounts


,count,mean,std,min,25%,50%,75%,max
balance_usd,75000.00,99921.85,57751.58,13.67,49955.56,99871.32,149960.86,199999.79



📊 TABLA: Customers


,count,mean,std,min,25%,50%,75%,max
credit_score,50000.00,575.13,159.14,300.00,437.00,576.00,713.00,850.00



📊 TABLA: Loans


,count,mean,std,min,25%,50%,75%,max
loan_amount,30000.00,150436.66,86162.48,1015.54,76351.80,149801.65,225473.12,299996.69
interest_rate,30000.00,8.54,3.75,2.00,5.31,8.56,11.80,15.00



📊 TABLA: Transactions


,count,mean,std,min,25%,50%,75%,max
amount_usd,1000000.00,5001.16,2887.41,1.02,2502.80,5000.73,7503.94,9999.98


### Conclusión

Los estadísticos descriptivos muestran que las variables numéricas presentan rangos de valores coherentes con la naturaleza del negocio.

No se han identificado valores negativos donde no deberían existir ni registros con valores fuera de los dominios esperados. Variables como el saldo de las cuentas, el importe de los préstamos, los importes de las transacciones y la puntuación crediticia mantienen distribuciones consistentes con el contexto del proyecto.

A partir de esta revisión se considera que las variables numéricas presentan una calidad adecuada para continuar con el análisis estadístico.

# 10. Conclusiones generales

El proceso de validación realizado sobre la base de datos permite concluir que el modelo de datos presenta un nivel de calidad adecuado para continuar con las siguientes fases del proyecto.

Las principales comprobaciones realizadas han sido:

- Se estableció correctamente la conexión con la base de datos SQLite.
- Todas las tablas fueron cargadas sin incidencias.
- La estructura del modelo relacional es consistente y los tipos de datos son adecuados para el análisis.
- No se detectaron valores nulos, salvo en las columnas **city** y **country** de la tabla **Branches**, donde el 100 % de los registros carecen de información. Estas columnas se evaluarán en función de su relevancia para el proyecto.
- No se identificaron registros duplicados en ninguna de las tablas.
- La validación de la integridad referencial confirmó la ausencia de registros huérfanos entre las tablas relacionadas.
- Los estadísticos descriptivos muestran que las variables numéricas presentan rangos de valores coherentes y no se han detectado valores fuera de los dominios esperados.

En conjunto, los resultados obtenidos permiten considerar que la base de datos dispone de la calidad necesaria para iniciar la fase de análisis estadístico y generación de nuevas variables, objetivo del siguiente notebook del proyecto.